# Effects of Occlusion on Model Performance and Mitigation of These Effects

**A Neuromatch Academy project** — cookiecutter notebook based on the NMA *Example Deep Learning Project* template.

Fill in / adapt each step as the project progresses.

---
# Objectives

We're interested in how **occlusion** (partially hiding an object) affects the performance of image-classification models, and whether **data augmentation** can mitigate these effects. We will use the mammal classes of CIFAR-100, introduce simulated occlusions of different sizes and types (Cutout-style blocks, GridMask), and compare a simple CNN against ResNet18 — trained with and without augmentation.

**Questions (from the project doc):**
1. How does occlusion affect model performance in different types of models?
2. How do the size and type of occlusion contribute to this?
3. How can the effects of occlusion be mitigated?

**Resources**:
* [Paper close to our idea (the one Mohamed sent)](https://arxiv.org/pdf/2409.10775)
* [Occlusion Detection and Handling: A Review](https://www.researchgate.net/publication/278729405_Occlusion_Detection_and_Handling_A_Review)
* [Survey of data-augmentation techniques](https://ieeexplore.ieee.org/abstract/document/10699340)
* [CIFAR-100 dataset](https://www.cs.toronto.edu/~kriz/cifar.html)

In [ ]:
%pip install torch torchvision numpy scikit-learn matplotlib

---
# Setup

As in the NMA template, we keep the setup cells at the top: Imports, Figure settings, Plotting functions, and Data retrieval.

In [ ]:
# Imports
# matrices and plotting:
import numpy as np
import matplotlib.pyplot as plt

# pytorch:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# torchvision for CIFAR-100, transforms and ResNet18:
import torchvision
from torchvision import transforms
from torchvision.models import resnet18

# metrics from sklearn:
from sklearn.metrics import confusion_matrix, precision_score

# to get some idea of how long stuff will take to complete:
import time

# to see how (un)balanced the data is:
from collections import Counter

In [ ]:
# @title Figure settings
%config InlineBackend.figure_format = 'retina'
plt.style.use("https://raw.githubusercontent.com/NeuromatchAcademy/content-creation/main/nma.mplstyle")

In [ ]:
# @title Plotting functions

def plotConfusionMatrix(real_labels, predicted_labels, label_names):

  # convert the labels to integers:
  real_labels = [int(x) for x in real_labels]
  predicted_labels = [int(x) for x in predicted_labels]
  tick_names = [a.replace("_", " ") for a in label_names]

  cm = confusion_matrix(real_labels, predicted_labels, normalize='true')

  fig = plt.figure(figsize=(10, 8))
  plt.imshow(cm)
  plt.xticks(range(len(tick_names)), tick_names, rotation=90)
  plt.yticks(range(len(tick_names)), tick_names)
  plt.xlabel('predicted class')
  plt.ylabel('real class')
  plt.colorbar()
  plt.show()


def showImageGrid(images, titles=None, nrow=1, ncol=None):
  """Show a row/grid of uint8 HxWxC images."""
  n = len(images)
  ncol = ncol or n
  fig, axes = plt.subplots(nrow, ncol, figsize=(2 * ncol, 2 * nrow))
  axes = np.atleast_1d(axes).ravel()
  for i, ax in enumerate(axes):
    if i < n:
      ax.imshow(images[i])
      if titles is not None:
        ax.set_title(titles[i], fontsize=9)
    ax.axis('off')
  plt.tight_layout()
  plt.show()

In [ ]:
# @title Data retrieval
# @markdown Run this cell to download CIFAR-100 and keep only the mammal classes.

# The 25 mammal classes of CIFAR-100 (5 mammal superclasses x 5 classes):
MAMMAL_CLASSES = [
    # aquatic mammals
    'beaver', 'dolphin', 'otter', 'seal', 'whale',
    # large carnivores
    'bear', 'leopard', 'lion', 'tiger', 'wolf',
    # large omnivores and herbivores
    'camel', 'cattle', 'chimpanzee', 'elephant', 'kangaroo',
    # medium-sized mammals
    'fox', 'porcupine', 'possum', 'raccoon', 'skunk',
    # small mammals
    'hamster', 'mouse', 'rabbit', 'shrew', 'squirrel',
]

cifar_train = torchvision.datasets.CIFAR100(root='./data', train=True, download=True)
cifar_test = torchvision.datasets.CIFAR100(root='./data', train=False, download=True)

label_names = MAMMAL_CLASSES
mammal_ids = [cifar_train.class_to_idx[c] for c in MAMMAL_CLASSES]
old_to_new = {old: new for new, old in enumerate(mammal_ids)}

def filterMammals(ds):
  images, labels = [], []
  for img, lbl in zip(ds.data, ds.targets):
    if lbl in old_to_new:
      images.append(img)
      labels.append(old_to_new[lbl])
  return np.stack(images), np.array(labels)

train_images, train_labels = filterMammals(cifar_train)
test_images, test_labels = filterMammals(cifar_test)

print('train:', train_images.shape, '  test:', test_images.shape)

---
# Step 1: Question

**"How does occlusion affect the performance of image-classification models, how do the size and type of the occlusion contribute, and can data augmentation mitigate the effect?"**

Our goal is a *pilot* study: introduce simulated occlusion into CIFAR-100 mammal images, quantify the performance drop for a simple CNN and for ResNet18, and test whether augmentation (including occlusion-like augmentations such as Random Erasing) makes the models more robust.

Possible extensions (from the project doc): Faster R-CNN on regular / occluded / augmented data, effects of masks or segmentation and other models (e.g. YOLO), and occlusion of specific body parts using DeepLabCut.

---
# Step 2: literature review

Most importantly, our literature review needs to address the following:

1. **What techniques help to reduce the effects of occlusion?**
   * https://arxiv.org/pdf/2409.10775 (the one that Mohamed sent)
2. **What are the types of occlusion?**
   * *Naturally occurring*: self-occlusion, inter-object occlusion, scene occlusion, object truncation.
   * *Simulated*: deliberately introduced for training/augmentation —
     * single-region synthetic block removal: **Cutout**, **Random Erasing**
     * multi-region structured removal: **GridMask**, **Hide-and-Seek**
3. **What types of occlusion have the most severe consequences for computer-vision models?**
   * [Occlusion Detection and Handling: A Review](https://www.researchgate.net/publication/278729405_Occlusion_Detection_and_Handling_A_Review)
4. **What types of augmentation strategies could we use?**
   * Survey: https://ieeexplore.ieee.org/abstract/document/10699340
   * *Geometric*: rotation, translation, shearing, flipping
   * *Non-geometric*: cropping/resizing, noise injection, color space, jitter, kernel filters
   * *Image erasing*: Cutout, Random Erasing, **Hide-and-Seek** (particularly effective against occlusion), GridMask
   * *Advanced*: LocalAugment, SelfAugmentation, SalfMix, KeepAugment, YOCO, Cut-Thumbnail, Mixup, ...
5. The effects of occlusion on the (human) visual system and how occlusion reasoning develops.

---
# Step 3: ingredients

## Data ingredients

After the data retrieval cell we have 4 numpy arrays plus label names:

- `train_images`: 12500 training images (25 mammal classes x 500), shape 32x32x3, uint8
- `train_labels`: class labels (0-24) for the training images
- `test_images`: 2500 test images (25 x 100)
- `test_labels`: class labels for the test images
- `label_names`: text names of the 25 mammal classes

Let's have a quick look at the data:

In [ ]:
print(train_images.shape, train_images.dtype)
print(test_images.shape)

# class balance:
print(Counter(train_labels))
print(Counter(test_labels))

In [ ]:
# show one example of each of the first 8 classes:
examples = [train_images[np.where(train_labels == c)[0][0]] for c in range(8)]
showImageGrid(examples, titles=label_names[:8])

## Occlusion ingredients

We simulate occlusion at the pixel level. Sizes are defined on the 32x32 CIFAR images:

- **small**: 8x8 block (~6% of pixels)
- **medium**: 12x12 block (~14% of pixels)
- **large**: 16x16 block (~25% of pixels)

Types:
- **block** (Cutout-style): one random square region filled with a constant value
- **gridmask**: a regular grid of masked cells (structured, multi-region)

In [ ]:
# @title Occlusion functions

OCCLUSION_SIZES = {'small': 8, 'medium': 12, 'large': 16}  # block edge in pixels on 32x32 images

def occludeBlock(img, size=8, fill=0, rng=None):
  """Cutout-style occlusion: one random square region filled with a constant.

  Args:
    img (ndarray): HxWxC uint8 image.
    size (int): edge length of the square occlusion.
    fill (int): fill value (0=black, 255=white, or e.g. 128=gray).
    rng: optional numpy random Generator for reproducibility.
  """
  rng = rng if rng is not None else np.random.default_rng()
  img = img.copy()
  h, w = img.shape[:2]
  y = int(rng.integers(0, h - size + 1))
  x = int(rng.integers(0, w - size + 1))
  img[y:y + size, x:x + size, :] = fill
  return img


def occludeGridMask(img, cell=8, fill=0):
  """GridMask-style occlusion: mask alternating cells of a regular grid."""
  img = img.copy()
  h, w = img.shape[:2]
  for y in range(0, h, cell * 2):
    for x in range(0, w, cell * 2):
      img[y:y + cell, x:x + cell, :] = fill
  return img


def occludeImage(img, occlusion=None, rng=None):
  """Dispatch: occlusion in {None, 'small', 'medium', 'large', 'gridmask'}."""
  if occlusion is None:
    return img
  if occlusion == 'gridmask':
    return occludeGridMask(img)
  return occludeBlock(img, size=OCCLUSION_SIZES[occlusion], rng=rng)

Let's make sure the occlusion functions do what we intended:

In [ ]:
rng = np.random.default_rng(0)
img = train_images[0]

variants = [img,
            occludeImage(img, 'small', rng),
            occludeImage(img, 'medium', rng),
            occludeImage(img, 'large', rng),
            occludeImage(img, 'gridmask')]
showImageGrid(variants, titles=['original', 'small', 'medium', 'large', 'gridmask'])

## Model ingredients

**"Mechanisms"**:

* Occlusion? --> Simulated, pixel-level, applied on-the-fly in the dataset object so every epoch sees different occlusion locations. We vary **size** (small/medium/large) and **type** (block/gridmask).

* Augmentation (the mitigation)? --> Standard geometric augmentation (random crop with padding, horizontal flip) plus **Random Erasing**, an occlusion-like augmentation which the literature suggests improves robustness to occlusion.

* Classifiers? --> A simple CNN (baseline) and **ResNet18** (deeper, with skip connections). Both trained from scratch on the same data.

* Input? --> 32x32x3 CIFAR-100 mammal images, normalized; occluded or not, augmented or not.

* Output? --> 25 class probabilities; we take the argmax and evaluate with **accuracy, per-class precision and the confusion matrix**.

---
# Step 4: hypotheses

* **Hypothesis 1**: Occlusion reduces model performance, visible as more confusion in the confusion matrix and lower precision; larger occlusions hurt more.
  * *Null hypothesis*: occlusion does not change performance (no change in confusion matrix or precision).

* **Hypothesis 2**: Augmentation reduces the effect of occlusion on model performance (recovers part of the lost precision / cleans up the confusion matrix).
  * *Null hypothesis*: augmentation does not change performance on occluded data.

* **Hypothesis 3**: ResNet18 performs better than the simple CNN, even with occlusion.
  * *Null hypothesis*: the models perform equally well.

In mathematical terms:
* Hypothesis 1: $\mathbb{E}(perf_{clean}) > \mathbb{E}(perf_{occluded})$, and $perf$ decreases with occlusion size
* Hypothesis 2: $\mathbb{E}(perf_{occluded+aug}) > \mathbb{E}(perf_{occluded})$
* Hypothesis 3: $\mathbb{E}(perf_{ResNet18}) > \mathbb{E}(perf_{CNN})$ for every condition

---
# Step 5: toolkit selection

We need a toolkit that can classify images. We keep it as simple as possible:

* **PyTorch + torchvision**: CIFAR-100 download, transforms (augmentation), and a reference ResNet18 implementation.
* A small **2D convnet** written by us as the baseline model.
* **scikit-learn** for the confusion matrix and precision scores.

This allows us to answer our question (performance under occlusion), speaks to our hypotheses (model comparison, augmentation mitigation), and is feasible in NMA project time. Training is much faster on a GPU — in Colab, set `Runtime > Change runtime type > GPU`.

---
# Step 6: model drafting

Our sketch of the experiment pipeline:

```
                 +--------------------+
CIFAR-100        |  occlusion (none / |        +-----------+
mammals   -----> |  small / medium /  | -----> | augment?  | -----> CNN or ResNet18 -----> 25 classes
(32x32x3)        |  large / gridmask) |        | (yes/no)  |
                 +--------------------+        +-----------+
```

Experiment grid (Steps 1-5 of the project doc):

| # | model    | training data | augmentation |
|---|----------|---------------|--------------|
| 1 | CNN      | regular       | no           |
| 2 | CNN      | occluded      | no           |
| 3 | ResNet18 | regular       | no           |
| 4 | ResNet18 | occluded      | no           |
| 5a| CNN      | occluded      | yes          |
| 5b| ResNet18 | occluded      | yes          |

Each trained model is evaluated on **both** the clean and the occluded test set.

---
# Step 7: model implementation

It's finally time to write some deep learning code... so here we go!

The cell below creates a `Dataset` object class (based on the PyTorch data tutorial, like in the NMA example project). We can tell the dataset object to use the training or test data, whether to apply **occlusion** (and which kind), and whether to apply **augmentation**:

In [ ]:
# normalization constants for CIFAR-100:
CIFAR_MEAN = (0.5071, 0.4865, 0.4409)
CIFAR_STD = (0.2673, 0.2564, 0.2762)


class OccludedMammalDataset(Dataset):
  """CIFAR-100 mammals with optional on-the-fly occlusion and augmentation."""

  def __init__(self, train=True, occlusion=None, augment=False, seed=None):
    """
    Args:
      train (boolean): Use the training data, or otherwise the test data.
      occlusion (str|None): None, 'small', 'medium', 'large' or 'gridmask'.
      augment (boolean): Apply augmentation (flip + crop + random erasing).
      seed (int|None): Seed for reproducible occlusion locations (use for test sets).
    """
    if train:
      self.images = train_images
      self.labels = train_labels
    else:
      self.images = test_images
      self.labels = test_labels

    self.occlusion = occlusion
    self.rng = np.random.default_rng(seed)

    normalize = [transforms.ToTensor(),
                 transforms.Normalize(CIFAR_MEAN, CIFAR_STD)]
    if augment:
      self.transform = transforms.Compose([
          transforms.ToPILImage(),
          transforms.RandomHorizontalFlip(),
          transforms.RandomCrop(32, padding=4),
          *normalize,
          transforms.RandomErasing(p=0.5),  # occlusion-like augmentation
      ])
    else:
      self.transform = transforms.Compose(normalize)

  def __len__(self):
    return self.labels.size

  def __getitem__(self, idx):
    if torch.is_tensor(idx):
      idx = idx.tolist()

    img = occludeImage(self.images[idx], self.occlusion, rng=self.rng)
    return self.transform(img), int(self.labels[idx])

We want to make sure that this object works the way we intended, so we try it out:

In [ ]:
mammals_train = OccludedMammalDataset(train=True)
mammals_test_occ = OccludedMammalDataset(train=False, occlusion='medium', seed=42)

print('TRAINING:', len(mammals_train))
x, y = mammals_train[0]
print(x.shape, label_names[y])

print('\nTESTING (occluded):', len(mammals_test_occ))
x, y = mammals_test_occ[0]
print(x.shape, label_names[y])

## Build models

PyTorch expects a minibatch of B samples as input, so a 2D CNN gets a 4D tensor: **BxCxHxW**

- **B:** batch size
- **C:** channels (3 color channels)
- **H, W:** height and width (32x32)

First the simple baseline CNN (2 conv blocks + 2 fully connected layers), then a helper that builds a fresh (not pretrained) ResNet18 adapted to 32x32 inputs:

In [ ]:
class MammalCNN(nn.Module):
  """Simple baseline 2D convnet for 32x32 images, 25 classes."""

  def __init__(self, num_classes=25):
    super(MammalCNN, self).__init__()

    self.layer1 = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2))  # -> 32x16x16

    self.layer2 = nn.Sequential(
        nn.Conv2d(32, 64, kernel_size=3, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=2, stride=2))  # -> 64x8x8

    self.dropout1 = nn.Dropout(p=0.2)
    self.fc1 = nn.Linear(64 * 8 * 8, 512)
    self.nl = nn.ReLU()
    self.dropout2 = nn.Dropout(p=0.2)
    self.fc2 = nn.Linear(512, num_classes)

  def forward(self, x):
    out = self.layer1(x)
    out = self.layer2(out)

    out = out.reshape(out.size(0), -1)
    out = self.dropout1(out)
    out = self.fc1(out)
    out = self.nl(out)
    out = self.dropout2(out)
    out = self.fc2(out)
    out = nn.functional.log_softmax(out, dim=1)

    return out


def makeResNet18(num_classes=25):
  """ResNet18 from scratch, adapted for 32x32 CIFAR images."""
  model = resnet18(weights=None, num_classes=num_classes)
  # CIFAR images are small: use a 3x3 first conv and drop the first maxpool
  model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
  model.maxpool = nn.Identity()
  return model

We can now instantiate a model, and set a criterion and optimizer:

In [ ]:
### ADDING GPU ###
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f"using device: {device}")

# Hyperparameters
num_epochs = 10       # increase (e.g. 30+) for real runs
batch_size = 128
learning_rate = 0.001

# Create training and test datasets and loaders (condition 1: CNN on regular data)
mammals_train = OccludedMammalDataset(train=True)
mammals_test = OccludedMammalDataset(train=False)

train_loader = DataLoader(dataset=mammals_train, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(dataset=mammals_test, batch_size=batch_size, shuffle=False)

# create the model object:
model = MammalCNN(num_classes=25).to(device)

# loss and optimizer:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

And now we are ready to train this model.

**This takes a few minutes on CPU, well under a minute on GPU.**

In [ ]:
# Train the model
total_step = len(train_loader)
loss_list = []
acc_list = []
for epoch in range(num_epochs):
  for i, (images, labels) in enumerate(train_loader):
    images, labels = images.to(device), labels.to(device)

    # Run the forward pass
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss_list.append(loss.item())

    # Backprop and perform Adam optimisation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Track the accuracy
    total = labels.size(0)
    _, predicted = torch.max(outputs.data, 1)
    correct = (predicted == labels).sum().item()
    acc_list.append(correct / total)

  print(f"Epoch [{epoch+1}/{num_epochs}], "
        f"Loss: {loss.item():.4f}, "
        f"Accuracy: {((correct / total) * 100):.2f}%")

Chance level is $100/25 = 4\%$, so anything well above that means the model is learning. Now we want to see performance on the test data set — clean **and** occluded:

In [ ]:
# Test the model on the clean and on an occluded test set
def evaluateModel(model, test_loader):
  model.eval()
  real_labels, predicted_labels = [], []
  with torch.no_grad():
    correct = 0
    total = 0
    for images, labels in test_loader:
      images, labels = images.to(device), labels.to(device)
      real_labels += list(labels.cpu())
      outputs = model(images)
      _, predicted = torch.max(outputs.data, 1)
      predicted_labels += list(predicted.cpu())
      total += labels.size(0)
      correct += (predicted == labels).sum().item()

  accuracy = (correct / total) * 100
  precision = precision_score(real_labels, predicted_labels,
                              average='macro', zero_division=0) * 100
  return {'accuracy': accuracy,
          'precision': precision,
          'real_labels': real_labels,
          'predicted_labels': predicted_labels}


clean_results = evaluateModel(model, test_loader)
print(f"CLEAN test:    accuracy {clean_results['accuracy']:.2f}%, "
      f"macro precision {clean_results['precision']:.2f}%")

occluded_loader = DataLoader(OccludedMammalDataset(train=False, occlusion='medium', seed=42),
                             batch_size=batch_size, shuffle=False)
occ_results = evaluateModel(model, occluded_loader)
print(f"OCCLUDED test: accuracy {occ_results['accuracy']:.2f}%, "
      f"macro precision {occ_results['precision']:.2f}%")

Let's look at where the errors are:

In [ ]:
# plt.style.use('dark_background')  # uncomment if you're using dark mode...
plotConfusionMatrix(occ_results['real_labels'], occ_results['predicted_labels'], label_names)

---
# Step 8: Modeling completion

Are we done yet? In order to answer our questions, reach our goals and evaluate our hypotheses we need to run each **model** (CNN / ResNet18) under each **training condition** (regular / occluded, with / without augmentation) and evaluate on clean and occluded test sets. We wrap the whole train+test procedure in one function:

In [ ]:
def testOcclusionModel(model_name='cnn',
                       train_occlusion=None,
                       augment=False,
                       test_occlusions=(None, 'small', 'medium', 'large', 'gridmask'),
                       num_epochs=10,
                       batch_size=128,
                       learning_rate=0.001,
                       verbose=True):
  """Train one model in one condition and evaluate on several test sets.

  Args:
    model_name (str): 'cnn' or 'resnet18'.
    train_occlusion (str|None): occlusion applied to the training images.
    augment (boolean): apply augmentation during training.
    test_occlusions (tuple): occlusion conditions to evaluate on.
  """

  # Create training dataset and loader:
  ds_train = OccludedMammalDataset(train=True, occlusion=train_occlusion, augment=augment)
  train_loader = DataLoader(dataset=ds_train, batch_size=batch_size, shuffle=True)

  # create the model object:
  if model_name == 'cnn':
    model = MammalCNN(num_classes=25).to(device)
  elif model_name == 'resnet18':
    model = makeResNet18(num_classes=25).to(device)
  else:
    raise ValueError(f"unknown model: {model_name}")

  # loss and optimizer:
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

  # Train the model
  model.train()
  for epoch in range(num_epochs):
    for images, labels in train_loader:
      images, labels = images.to(device), labels.to(device)

      outputs = model(images)
      loss = criterion(outputs, labels)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  # Test the model on each test condition:
  results = {}
  for test_occ in test_occlusions:
    ds_test = OccludedMammalDataset(train=False, occlusion=test_occ, seed=42)
    test_loader = DataLoader(dataset=ds_test, batch_size=batch_size, shuffle=False)
    results[str(test_occ)] = evaluateModel(model, test_loader)
    if verbose:
      r = results[str(test_occ)]
      print(f"  test occlusion={str(test_occ):>8}: "
            f"accuracy {r['accuracy']:.2f}%, precision {r['precision']:.2f}%")

  return {'model': model, 'results': results}

Let's test this on one quick condition:

**Each call trains a full model — a few minutes on CPU, faster on GPU.**

In [ ]:
fit = testOcclusionModel(model_name='cnn', train_occlusion=None, augment=False,
                         test_occlusions=(None, 'medium'), num_epochs=5)
plotConfusionMatrix(fit['results']['medium']['real_labels'],
                    fit['results']['medium']['predicted_labels'],
                    label_names)

* Can we answer our question? --> YES, we can measure the performance drop caused by occlusion.
* Have we reached our goals? --> Almost — we still need the full comparison grid.
* Can we evaluate our hypotheses? --> YES, we can now run all conditions and compare them.

Good news, looks like we're done with a first iteration of modeling!

---
# Step 9: Model evaluation

We can now run the full experiment grid from the project doc (Steps 1-5): both models, trained on regular vs occluded data, without and with augmentation — always evaluated on clean and occluded test sets.

**This trains 8 models — expect it to take a while! (Use a GPU, and/or lower `num_epochs` for a quick pass.)**

In [ ]:
conditions = {
    '1: CNN / regular':            dict(model_name='cnn',      train_occlusion=None,     augment=False),
    '2: CNN / occluded':           dict(model_name='cnn',      train_occlusion='medium', augment=False),
    '3: ResNet18 / regular':       dict(model_name='resnet18', train_occlusion=None,     augment=False),
    '4: ResNet18 / occluded':      dict(model_name='resnet18', train_occlusion='medium', augment=False),
    '5a: CNN / occluded + aug':    dict(model_name='cnn',      train_occlusion='medium', augment=True),
    '5b: ResNet18 / occluded + aug': dict(model_name='resnet18', train_occlusion='medium', augment=True),
}

condition_fits = {}
for name, cfg in conditions.items():
  print(f"\n*** FITTING: {name}")
  t0 = time.time()
  condition_fits[name] = testOcclusionModel(test_occlusions=(None, 'medium'), **cfg)
  print(f"  ({time.time() - t0:.0f} s)")

And how does **occlusion size and type** matter (question 2)? We take one trained model and stress-test it across all occlusion conditions:

In [ ]:
print('Effect of occlusion size/type on the CNN trained on regular data:')
size_sweep = testOcclusionModel(model_name='cnn', train_occlusion=None, augment=False,
                                test_occlusions=(None, 'small', 'medium', 'large', 'gridmask'))

In [ ]:
# summary table of the full grid:
print(f"{'condition':<32} {'clean acc':>10} {'occl acc':>10} {'occl prec':>10}")
for name, fit in condition_fits.items():
  r = fit['results']
  print(f"{name:<32} {r['None']['accuracy']:>9.2f}% {r['medium']['accuracy']:>9.2f}% "
        f"{r['medium']['precision']:>9.2f}%")

Things to look for (our hypotheses):

* **H1**: within each row, is the occluded-test accuracy lower than the clean-test accuracy? Does the drop grow with occlusion size in the sweep?
* **H2**: do the `+ aug` rows recover accuracy/precision on the occluded test set compared to their non-augmented counterparts?
* **H3**: does ResNet18 beat the CNN in every condition?

For a formal test, fit each condition several times (different seeds) and compare medians — single runs are noisy.

---
# Step 10: publication

Let's write a simple abstract following the guidelines...

**A. What is the phenomena?** Here summarize the part of the phenomena which your modeling addresses.

*Occlusion — the partial obstruction of an object's visible surface — deprives vision models of evidence and degrades recognition.*

**B. What is the key scientific question?** Clearly articulate the question which your modeling tries to answer.

*Here, we ask how occlusion size and type affect the performance of image-classification models, and whether data augmentation mitigates these effects.*

**C. What was our hypothesis?** Explain the key relationships which we relied on to simulate the phenomena.

*We hypothesized that occlusion degrades performance (more so for larger occlusions), that augmentation reduces this degradation, and that ResNet18 outperforms a simple CNN even under occlusion.*

**D. How did your modeling work?** Give an overview of the model, its main components, and how the modeling works.

*We trained a simple CNN and a ResNet18 on the mammal classes of CIFAR-100, with and without simulated occlusion (Cutout-style blocks of varying size, GridMask) and with and without augmentation (geometric transforms and Random Erasing), and evaluated confusion matrices and precision on clean and occluded test sets.*

**E. What did you find? Did the modeling work?** Explain the key outcomes of your modeling evaluation.

*TODO — fill in after running Step 9.*

**F. What can you conclude?** Conclude as much as you can with reference to the hypothesis, within the limits of the modeling.

*TODO*

**G. What are the limitations and future directions?** What is left to be learned?

*Our occlusions are synthetic and may not capture natural occlusion statistics; CIFAR images are small (32x32); results may be architecture-specific. Future work: Faster R-CNN / YOLO on occluded detection tasks, segmentation masks, and body-part-specific occlusion with DeepLabCut.*

>Put this all in one paragraph (without the letters) for the final abstract.